# Clase 21 — Pandas aplicado

Este notebook profundiza en **navegación y estructura** de tablas en pandas: tipos de tabla, índice, selección con `.loc` e `.iloc`, y renombrado de columnas.

**Caso:** el equipo de soporte de **TiendaOnline** analiza los **tickets de mesa de ayuda** del **Q1 2026** (post-venta).

**Cómo usarlo:** ejecutá las celdas en orden; completá los ejercicios donde aparece `# Tu código aquí`.

### Objetivos de la clase

1. Reconocer **tipos de tabla** (principal, auxiliar, configuración) y decidir por dónde empezar el análisis.
2. Entender el **índice** de un DataFrame y cuándo reemplazarlo.
3. Seleccionar datos con **`.loc`** (etiquetas) e **`.iloc`** (posiciones).
4. Usar **`set_index`** y **`reset_index`** para reorganizar filas.
5. **Renombrar columnas** con `.rename()` y `df.columns`.

### Requisitos

- Entorno **`.venv`** del proyecto con **pandas** instalado (ver Clase 15 si necesitás crearlo).
- Archivos en la carpeta `datos/`: `tickets_soporte_q1.csv`, `clientes_soporte.csv`, `estados_ticket.csv`.
- Kernel **Python (.venv)** seleccionado en VS Code o Cursor.


## 0. Contexto — TiendaOnline, mesa de ayuda

**TiendaOnline** opera en **AR**, **CL** y **MX** por **Web**, **Tienda** y **Marketplace**. Tras la venta, el área de soporte registra **tickets** (consultas, reclamos, incidencias técnicas).

Antes de transformar datos, conviene reconocer **qué tablas** componen el escenario y **qué función** cumple cada una.

| Tipo | Función |
|------|---------|
| **Principal** | Contiene los registros centrales del proceso analizado |
| **Auxiliar** | Aporta información de apoyo para operaciones específicas |
| **Configuración** | Define parámetros, estados o categorías del sistema |

**Lectura inicial de tablas**

- La tabla **principal** define el punto de partida del análisis.
- Las **auxiliares** amplían contexto cuando aportan información necesaria (por ejemplo, segmento del cliente).
- Las de **configuración** ayudan a interpretar reglas, estados o categorías (por ejemplo, qué significa cada código de estado).

En esta clase cargamos tres archivos CSV que representan esos roles.


## 1. Librerías y rutas


In [2]:
import pandas as pd
from pathlib import Path

DATOS = Path("datos")

print("Pandas:", pd.__version__)


Pandas: 3.0.3


## 2. Tipos de tabla y lectura inicial

Identificamos el rol de cada tabla y la cargamos con `pd.read_csv()`.


In [3]:
# Tabla principal: un ticket por fila
tickets = pd.read_csv(DATOS / "tickets_soporte_q1.csv")

# Tabla auxiliar: datos de clientes (referencia)
clientes = pd.read_csv(DATOS / "clientes_soporte.csv")

# Tabla de configuración: significado de códigos de estado
estados = pd.read_csv(DATOS / "estados_ticket.csv")

print("Principal (tickets):", len(tickets), "filas,", len(tickets.columns), "columnas")
print("Auxiliar (clientes):", len(clientes), "filas")
print("Configuración (estados):", len(estados), "filas")


Principal (tickets): 50 filas, 10 columnas
Auxiliar (clientes): 15 filas
Configuración (estados): 5 filas


In [4]:
# Vista rápida de las tablas de apoyo
print("--- Clientes (auxiliar) ---")
display(clientes.head())

print("--- Estados (configuración) ---")
display(estados)


--- Clientes (auxiliar) ---


,id_cli,nombre_cliente,segmento
0,100,María González,Premium
1,101,Carlos Ruiz,Retail
2,102,Ana Martínez,Premium
3,103,Luis Fernández,Retail
4,104,Sofía López,Corporate


--- Estados (configuración) ---


,st_cod,estado_visible,es_final
0,1,Nuevo,False
1,2,Asignado,False
2,3,En curso,False
3,4,Resuelto,True
4,5,Cerrado,True


> **Punto de partida:** el análisis comienza en `tickets`. Las otras tablas se consultan para interpretar códigos o, en clases futuras, para combinar información por `id_cli` o `st_cod`.

> En esta clase **no hacemos merge** entre tablas; solo identificamos su rol.


## 3. Exploración guiada de la tabla principal

En la Clase 15 ya viste `head()`, `info()` y `describe()`. Aquí los usamos de nuevo, pero con un objetivo distinto: **observar la estructura** y decidir cómo navegar el DataFrame (índice, renombres, selección).

| Método | Para qué sirve |
|--------|----------------|
| `df.head(n)` | Primeras filas (vista rápida) |
| `df.info()` | Cantidad de filas, tipos y valores no nulos |
| `df.describe()` | Resumen de columnas numéricas |
| `df.describe(include="all")` | Incluye también columnas de texto |


In [5]:
tickets.head(10)


,id_ticket,fec_apertura,id_cli,cat_incid,prio,st_cod,canal_orig,reg,hrs_res,nps
0,5001,2026-01-05,120,Entrega,Media,1,Web,CL,37.1,2.0
1,5002,2026-01-08,115,Entrega,Media,2,Tienda,MX,57.3,2.0
2,5003,2026-01-09,110,Producto,Alta,3,Web,AR,68.9,1.0
3,5004,2026-01-12,115,Producto,Baja,4,Tienda,CL,66.8,5.0
4,5005,2026-01-11,120,Facturación,Baja,5,Web,MX,34.1,1.0
5,5006,2026-01-15,125,Devolución,Alta,1,Web,CL,5.6,5.0
6,5007,2026-01-15,105,Devolución,Alta,2,Tienda,CL,26.6,4.0
7,5008,2026-01-16,107,Entrega,Baja,3,Marketplace,MX,51.7,1.0
8,5009,2026-01-20,101,Facturación,Alta,4,Marketplace,AR,44.0,5.0
9,5010,2026-01-22,103,Entrega,Media,5,Web,AR,24.3,3.0


In [6]:
tickets.info()


<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_ticket     50 non-null     int64  
 1   fec_apertura  50 non-null     str    
 2   id_cli        50 non-null     int64  
 3   cat_incid     50 non-null     str    
 4   prio          50 non-null     str    
 5   st_cod        50 non-null     int64  
 6   canal_orig    50 non-null     str    
 7   reg           50 non-null     str    
 8   hrs_res       47 non-null     float64
 9   nps           49 non-null     float64
dtypes: float64(2), int64(3), str(5)
memory usage: 4.0 KB


In [7]:
tickets.describe()


,id_ticket,id_cli,st_cod,hrs_res,nps
count,50.00000,50.000000,50.000000,47.000000,49.000000
mean,5025.50000,109.540000,3.000000,30.323404,3.204082
std,14.57738,8.933336,1.428571,19.708244,1.471671
min,5001.00000,100.000000,1.000000,0.700000,1.000000
25%,5013.25000,102.000000,2.000000,15.400000,2.000000
50%,5025.50000,107.500000,3.000000,28.900000,3.000000
75%,5037.75000,115.000000,4.000000,43.750000,4.000000
max,5050.00000,130.000000,5.000000,68.900000,5.000000


In [8]:
tickets.describe(include="all")


,id_ticket,fec_apertura,id_cli,cat_incid,prio,st_cod,canal_orig,reg,hrs_res,nps
count,50.00000,50,50.000000,50,50,50.000000,50,50,47.000000,49.000000
unique,NaN,45,NaN,6,4,NaN,3,3,NaN,NaN
top,NaN,2026-01-15,NaN,Producto,Alta,NaN,Web,CL,NaN,NaN
freq,NaN,2,NaN,12,17,NaN,24,18,NaN,NaN
mean,5025.50000,NaN,109.540000,NaN,NaN,3.000000,NaN,NaN,30.323404,3.204082
std,14.57738,NaN,8.933336,NaN,NaN,1.428571,NaN,NaN,19.708244,1.471671
min,5001.00000,NaN,100.000000,NaN,NaN,1.000000,NaN,NaN,0.700000,1.000000
25%,5013.25000,NaN,102.000000,NaN,NaN,2.000000,NaN,NaN,15.400000,2.000000
50%,5025.50000,NaN,107.500000,NaN,NaN,3.000000,NaN,NaN,28.900000,3.000000
75%,5037.75000,NaN,115.000000,NaN,NaN,4.000000,NaN,NaN,43.750000,4.000000


### Qué observamos (y qué haremos después)

Revisá la salida de las celdas anteriores y contrastá con este checklist:

- **`id_ticket`**: identificador único por fila → candidato natural para **índice** (`set_index`).
- **Nombres abreviados** (`fec_apertura`, `cat_incid`, `st_cod`, …) → conviene **renombrar** para claridad.
- **`hrs_res` y `nps`**: pueden tener valores faltantes → `info()` y `describe()` lo muestran.
- **`st_cod`**: número que se interpreta con la tabla `estados` (configuración).
- **`id_cli`**: enlace conceptual con la tabla `clientes` (auxiliar).

A partir de estas observaciones pasamos a trabajar el **índice**, la **selección** y el **renombrado**.


## 4. Índice de un DataFrame

Una vez cargada la tabla de trabajo, cada fila se identifica mediante el **índice**. Por defecto suele ser numérico (`0, 1, 2, …`), pero puede reemplazarse por una columna del dataset.

- Define las **etiquetas** que identifican cada fila.
- **No** se interpreta como una columna ordinaria (no aparece en `df.columns` después de `set_index`).
- Puede contener números, textos o fechas.


In [9]:
# Índice por defecto
print("Índice actual (primeras 5 etiquetas):", tickets.index[:5].tolist())
print("Columnas:", list(tickets.columns))


Índice actual (primeras 5 etiquetas): [0, 1, 2, 3, 4]
Columnas: ['id_ticket', 'fec_apertura', 'id_cli', 'cat_incid', 'prio', 'st_cod', 'canal_orig', 'reg', 'hrs_res', 'nps']


### Establecer una columna como índice

Cuando el índice por defecto no representa la unidad de análisis, `set_index()` lo reemplaza con los valores de una columna existente.

```python
df = df.set_index("id_registro")
```

- La columna elegida pasa a identificar las filas.
- Las filas pueden accederse por ese valor usando `.loc`.
- El resultado debe **reasignarse** para conservar el cambio.


In [10]:
# Copia de trabajo para no perder la tabla original sin índice
df = tickets.copy()

# Exploramos con nombres originales; el índice lo definimos en la sección 5 tras renombrar.
# Por ahora, solo mostramos el índice por defecto:
df.index


RangeIndex(start=0, stop=50, step=1)

## 5. Renombrar columnas

Antes de fijar el índice, conviene dejar **nombres claros** en las columnas que seguirán siendo columnas.

Cuando los nombres de origen no son claros o consistentes, `.rename()` los modifica sin alterar los datos.

```python
df = df.rename(columns={"columna_anterior": "columna_nueva"})
```

- Recibe un diccionario con los nombres a cambiar.
- Conserva las columnas no incluidas en el diccionario.
- El resultado debe **reasignarse**.


In [11]:
df = df.rename(columns={
    "fec_apertura": "fecha_apertura",
    "id_cli": "id_cliente",
    "cat_incid": "categoria",
    "prio": "prioridad",
    "st_cod": "codigo_estado",
    "canal_orig": "canal",
    "reg": "region",
    "hrs_res": "horas_resolucion",
    "nps": "satisfaccion",
})

df.head(3)


,id_ticket,fecha_apertura,id_cliente,categoria,prioridad,codigo_estado,canal,region,horas_resolucion,satisfaccion
0,5001,2026-01-05,120,Entrega,Media,1,Web,CL,37.1,2.0
1,5002,2026-01-08,115,Entrega,Media,2,Tienda,MX,57.3,2.0
2,5003,2026-01-09,110,Producto,Alta,3,Web,AR,68.9,1.0


### Reemplazar todos los nombres de columnas

`df.columns` permite reemplazar **todos** los nombres mediante una lista completa.

```python
df.columns = ["fecha", "cliente", "producto", "importe"]
```

- La lista debe tener un nombre por cada columna.
- Reemplaza todos los nombres a la vez.
- Se usa cuando la estructura de columnas ya está verificada.

En nuestro caso, `id_ticket` pasará al índice; las demás columnas ya quedaron claras con `.rename()`. Mostramos la lista actual:


In [12]:
df.columns


Index(['id_ticket', 'fecha_apertura', 'id_cliente', 'categoria', 'prioridad',
       'codigo_estado', 'canal', 'region', 'horas_resolucion', 'satisfaccion'],
      dtype='str')

### Establecer `id_ticket` como índice

Ahora sí usamos `set_index` con un nombre ya legible.


In [13]:
df = df.set_index("id_ticket")
df.head(3)


,fecha_apertura,id_cliente,categoria,prioridad,codigo_estado,canal,region,horas_resolucion,satisfaccion
id_ticket,,,,,,,,,
5001,2026-01-05,120,Entrega,Media,1,Web,CL,37.1,2.0
5002,2026-01-08,115,Entrega,Media,2,Tienda,MX,57.3,2.0
5003,2026-01-09,110,Producto,Alta,3,Web,AR,68.9,1.0


In [14]:
# id_ticket ya no está en df.columns
print("Índice:", df.index.name)
print("Columnas:", list(df.columns))


Índice: id_ticket
Columnas: ['fecha_apertura', 'id_cliente', 'categoria', 'prioridad', 'codigo_estado', 'canal', 'region', 'horas_resolucion', 'satisfaccion']


## 6. Selección por etiqueta — `.loc`

A partir del índice, **`.loc`** selecciona filas y columnas usando las **etiquetas** del índice y los **nombres** de las columnas.

| Sintaxis | Devuelve |
|----------|----------|
| `df.loc[etiqueta_fila]` | Una fila como Series |
| `df.loc[etiqueta_fila, "columna"]` | Un valor específico |
| `df.loc[:, "columna"]` | Una columna completa |
| `df.loc[lista_filas, lista_columnas]` | Un subconjunto del DataFrame |


In [15]:
# Una fila completa (ticket 5025)
df.loc[5025]


fecha_apertura      2026-02-16
id_cliente                 103
categoria             Producto
prioridad                 Alta
codigo_estado                5
canal                      Web
region                      CL
horas_resolucion          14.1
satisfaccion               3.0
Name: 5025, dtype: object

In [16]:
# Un valor específico
df.loc[5025, "prioridad"]


'Alta'

In [17]:
# Una columna completa (Series)
df.loc[:, "categoria"].head()


id_ticket
5001        Entrega
5002        Entrega
5003       Producto
5004       Producto
5005    Facturación
Name: categoria, dtype: str

In [18]:
# Subconjunto: varios tickets y columnas
df.loc[[5020, 5025, 5030], ["fecha_apertura", "prioridad", "region"]]


,fecha_apertura,prioridad,region
id_ticket,,,
5020,2026-02-07,Media,AR
5025,2026-02-16,Alta,CL
5030,2026-02-23,Alta,MX


### Ejercicio — `.loc`

Usá `.loc` para obtener el ticket **5033** y mostrar solo las columnas `categoria`, `prioridad` y `horas_resolucion`.


In [19]:
# Tu código aquí





## 7. Selección por posición — `.iloc`

Cuando el acceso se define por **ubicación**, **`.iloc`** selecciona filas y columnas usando **posiciones numéricas**, sin depender de las etiquetas del índice.

| Sintaxis | Devuelve |
|----------|----------|
| `df.iloc[0]` | La primera fila como Series |
| `df.iloc[0, 2]` | Valor de la primera fila y tercera columna |
| `df.iloc[:, 0]` | La primera columna completa |
| `df.iloc[0:5, 0:3]` | Primeras cinco filas y tres columnas |


In [20]:
# Primera fila del DataFrame actual
df.iloc[0]


fecha_apertura      2026-01-05
id_cliente                 120
categoria              Entrega
prioridad                Media
codigo_estado                1
canal                      Web
region                      CL
horas_resolucion          37.1
satisfaccion               2.0
Name: 5001, dtype: object

In [21]:
# Primera fila, tercera columna (posición 2)
df.iloc[0, 2]


'Entrega'

In [22]:
# Primera columna
df.iloc[:, 0].head()


id_ticket
5001    2026-01-05
5002    2026-01-08
5003    2026-01-09
5004    2026-01-12
5005    2026-01-11
Name: fecha_apertura, dtype: str

In [23]:
# Primeras 5 filas y 3 columnas
df.iloc[0:5, 0:3]


,fecha_apertura,id_cliente,categoria
id_ticket,,,
5001,2026-01-05,120,Entrega
5002,2026-01-08,115,Entrega
5003,2026-01-09,110,Producto
5004,2026-01-12,115,Producto
5005,2026-01-11,120,Facturación


### Contraste: `.loc` vs `.iloc` después de ordenar

Si ordenamos por `fecha_apertura`, la **primera fila** cambia de **etiqueta** en el índice, pero sigue siendo la **posición 0** para `.iloc`.


In [24]:
df_ordenado = df.sort_values("fecha_apertura")

print("Etiqueta de la primera fila (.loc posición 0):", df_ordenado.index[0])
print("Misma fila con .iloc[0] — índice:", df_ordenado.iloc[0].name)

# Acceso por etiqueta estable del ticket original
print("Ticket 5001 sigue accesible:", df_ordenado.loc[5001, "fecha_apertura"] if 5001 in df_ordenado.index else "N/A")


Etiqueta de la primera fila (.loc posición 0): 5001
Misma fila con .iloc[0] — índice: 5001
Ticket 5001 sigue accesible: 2026-01-05


## 8. Criterio de selección: `.loc` o `.iloc`

| Situación | Método conveniente |
|-----------|-------------------|
| Se conoce la etiqueta de la fila o el nombre de la columna | `.loc` |
| Se conoce la posición numérica de fila o columna | `.iloc` |
| Hay una etiqueta estable para identificar la fila | `.loc` |
| Se necesita acceder por rango de posiciones | `.iloc` |

**Mini quiz (respondé mentalmente o en una celda markdown):**

1. ¿Cómo obtendrías la prioridad del ticket `5040`?
2. ¿Cómo obtendrías la segunda fila y cuarta columna del DataFrame actual?


## 9. Reiniciar el índice — `reset_index()`

`reset_index()` devuelve el índice a una secuencia numérica y convierte el índice anterior en **columna**.

```python
df = df.reset_index()
```

- Útil después de **filtros** u **ordenamientos**.
- Permite recuperar el índice anterior como dato visible.
- `drop=True` descarta el índice anterior sin agregarlo como columna.


In [25]:
# Filtramos tickets críticos (el índice conserva id_ticket)
criticos = df[df["prioridad"] == "Crítica"]
print("Tickets críticos:", len(criticos))
criticos.head()


Tickets críticos: 7


,fecha_apertura,id_cliente,categoria,prioridad,codigo_estado,canal,region,horas_resolucion,satisfaccion
id_ticket,,,,,,,,,
5011,2026-01-21,130,Técnico,Crítica,1,Web,CL,27.3,5.0
5019,2026-02-04,120,Facturación,Crítica,4,Web,AR,28.9,4.0
5027,2026-02-19,101,Facturación,Crítica,2,Web,AR,34.2,4.0
5031,2026-02-26,109,Devolución,Crítica,1,Marketplace,MX,34.9,3.0
5041,2026-03-14,102,Entrega,Crítica,1,Tienda,MX,6.3,2.0


In [26]:
# reset_index: id_ticket vuelve a ser columna
criticos_con_indice = criticos.reset_index()
criticos_con_indice.head()


,id_ticket,fecha_apertura,id_cliente,categoria,prioridad,codigo_estado,canal,region,horas_resolucion,satisfaccion
0,5011,2026-01-21,130,Técnico,Crítica,1,Web,CL,27.3,5.0
1,5019,2026-02-04,120,Facturación,Crítica,4,Web,AR,28.9,4.0
2,5027,2026-02-19,101,Facturación,Crítica,2,Web,AR,34.2,4.0
3,5031,2026-02-26,109,Devolución,Crítica,1,Marketplace,MX,34.9,3.0
4,5041,2026-03-14,102,Entrega,Crítica,1,Tienda,MX,6.3,2.0


In [27]:
# drop=True: descarta el índice anterior (id_ticket) al reiniciar
criticos_reset = criticos.reset_index(drop=True)
criticos_reset.head()


,fecha_apertura,id_cliente,categoria,prioridad,codigo_estado,canal,region,horas_resolucion,satisfaccion
0,2026-01-21,130,Técnico,Crítica,1,Web,CL,27.3,5.0
1,2026-02-04,120,Facturación,Crítica,4,Web,AR,28.9,4.0
2,2026-02-19,101,Facturación,Crítica,2,Web,AR,34.2,4.0
3,2026-02-26,109,Devolución,Crítica,1,Marketplace,MX,34.9,3.0
4,2026-03-14,102,Entrega,Crítica,1,Tienda,MX,6.3,2.0


## 10. Ejercicio integrador

Completá el pipeline en la celda siguiente:

1. Cargar `tickets_soporte_q1.csv` en un DataFrame nuevo.
2. Mostrar `head(5)` e `info()`.
3. Renombrar al menos tres columnas con `.rename()`.
4. Establecer `id_ticket` como índice.
5. Con `.loc`, mostrar el ticket **5042** completo.
6. Con `.iloc`, mostrar las **primeras 3 filas** y **4 columnas**.
7. Filtrar filas con `region == "AR"`, luego aplicar `reset_index()` (sin `drop`).


In [28]:
# Tu código aquí





## 11. Cierre

En esta clase:

- Identificaste tablas **principal**, **auxiliar** y de **configuración**.
- Exploraste el dataset con `head`, `info` y `describe` antes de transformarlo.
- Renombraste columnas, definiste un **índice** con `set_index` y navegaste con **`.loc`** e **`.iloc`**.
- Reiniciaste el índice tras un filtro con **`reset_index`**.

**Próximo paso sugerido:** combinar `tickets` con `clientes` y `estados` usando `merge` (clase posterior).
